In [1]:
import json
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import os
from dotenv import load_dotenv
from openai import OpenAI
import requests

load_dotenv()

True

In [3]:
import pandas as pd

In [11]:
from pprint import pprint

In [2]:
# environment variables
api_key = os.getenv("OPENROUTER_API_KEY")
endpoint = os.getenv("ENDPOINT")

# chat client
chat_client = OpenAI(api_key=api_key, base_url=endpoint)

In [9]:
df = pd.read_csv("course_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   class_id             211 non-null    object
 1   course_name          211 non-null    object
 2   instructor           211 non-null    object
 3   course_type          211 non-null    object
 4   location             211 non-null    object
 5   cost                 211 non-null    object
 6   learning_objectives  211 non-null    object
 7   provided_materials   211 non-null    object
 8   skills_developed     211 non-null    object
 9   course_description   211 non-null    object
dtypes: object(10)
memory usage: 16.6+ KB


In [7]:
for _, i in df.head(3).iterrows():
    print(i.get("class_id"))

CLASS_4033
CLASS_0317
CLASS_8879


In [ ]:
# create a series of chunks in which to
chunks = []
for i, course in df.iterrows():
    name = course.get("course_name", "")
    instructor = course.get("instructor", "")
    course_type = course.get("course_type", "")
    location = course.get("location", "")
    cost = course.get("cost", "")
    learning_objectives = course.get("learning_objectives", "")
    materials = course.get("provided_materials", "")
    skills = course.get("skills_developed", "")
    description = course.get("course_description", "")

    # text that will be embedded
    text = (
        f"Name: {name}\n"
        f"Instructor: {instructor}\n"
        f"Course Type: {course_type}\n"
        f"Location: {location}\n"
        f"Cost: {cost}\n"
        f"Learning Objectives: {learning_objectives}\n"
        f"Provided Materials: {materials}\n"
        f"Skills Developed: {skills}\n"
        f"Description: {description}"
    )
    # Clean up the data for filtering.
    chunk = {
        "id": f"dandori_{i}_{name}",
        "text": text,
        "metadata": {
            "course_name": name,
            "instructor": instructor,
            "course_type": course_type,
            "location": location,
            "cost": float(cost) if cost else 0.0, # Convert to number for filtering
            "source": "dandori_db"
        },
    }
    chunks.append(chunk)

pprint(chunks[0])

{'id': 'dandori_0_The Art of Wondrous Waffle Weaving',
 'metadata': {'cost': 'price to attend the course in great british pounds',
              'course type': 'what category the course is in',
              'description': 'a description of the course goals and the '
                             'activities it involves',
              'instructor': 'name of the instructor running the course',
              'learning objectives': 'a list of goals of the course',
              'location': 'where the course is run',
              'name': 'name of the course',
              'provided materials': 'a list of materials provided by the '
                                    'instructor',
              'skills developed': 'a list of skills the course aims to '
                                  'develop'},
 'text': 'Name: The Art of Wondrous Waffle Weaving\n'
         'Instructor: Chef Waffleby\n'
         'Course Type: Culinary Arts\n'
         'Location: Harrogate\n'
         'Cost: £75.00\n'
 

In [38]:
# Create a Chroma client
client = chromadb.Client()

In [39]:
# Get or create collection
collection = client.get_or_create_collection(name="courses")

In [40]:
# Allow your client to use a different embedder
"""
Create a Class which will allow me to use my own embedder with ChromaDB
"""
# SOLUTION


class Embedder(EmbeddingFunction):
    """
    Chroma-compatible embedding function that calls OpenRouter's /embeddings API.
    """

    def __init__(
        self,
        api_key: str | None = None,
        model: str | None = None,
        base_url: str | None = None,
    ):
        self.base_url = endpoint or os.getenv("ENDPOINT")
        self.api_key = api_key or os.getenv("OPENROUTER_API_KEY")
        if not self.api_key:
            raise RuntimeError("Embedder: API_KEY not set.")
        self.model = model
        self.base_url = base_url

    def __call__(self, inputs: Documents) -> Embeddings:
        # Chroma passes a list of strings as `inputs`
        if not inputs:
            return []

        resp = requests.post(
            f"{self.base_url.rstrip('/')}/embeddings",
            headers={
                "Authorization": f"Bearer {self.api_key}",
            },
            json={
                "model": self.model,
                "input": list(inputs),
            },
            timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()
        # pprint(data)
        return [item["embedding"] for item in data["data"]]


collection = client.get_or_create_collection(
    name="courses_gem",
    embedding_function=Embedder(
        model="google/gemini-embedding-001", api_key=api_key, base_url=endpoint
    ),
)

In [20]:
os.getenv("ENDPOINT")

'https://openrouter.ai/api/v1'

In [41]:
# 5. Add all class documents

BATCH_SIZE = 50
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    collection.add(
        ids=[c.get("id") for c in batch],
        documents=[c.get("text") for c in batch],
        metadatas=[c.get("metadata") for c in batch],
    )
    print(f"Added batch {i//BATCH_SIZE + 1}")

# collection.add(
#     ids=[i.get("id") for i in chunks],
#     documents=[i.get("text") for i in chunks],
#     metadatas=[i.get("metadata") for i in chunks],
# )

Added batch 1
Added batch 2
Added batch 3
Added batch 4
Added batch 5
